In [60]:
# ---- DAT

In [7]:
from pathlib import Path
import importlib.util

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.animation import FFMpegWriter, PillowWriter
from astropy.io import fits
from tqdm import tqdm

# =========================================================
# CONFIG
# =========================================================

OUTPUT_FORMAT = "gif"        # "gif" or "mp4"
THEME = "dark"               # "dark" or "light"

ANIMATION_STYLE = "bottom_to_top"
# "left_to_right"  — линии рисуются слева направо
# "bottom_to_top"  — линии поднимаются снизу вверх
# "alive"          — линии сразу видны и слегка меняют амплитуду

RENDER_MODE = "single_files"
# "comparison"     — один общий график со всеми спектрами
# "single_files"   — один спектр = один файл

DATA_DIR = Path("data")
OUT_ROOT = Path("animations/spectra")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

FPS = 24
DPI = 160
FRAMES = 160

WAVELENGTH_RANGE = (6500, 6600)
LAMBDA_0 = 6499.99289377311
DELTA_LAMBDA = 0.03105132609471184

# =========================================================
# STYLE TOKENS
# =========================================================

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "style" / "hud_style_tokens.py").exists():
            return p
    return Path.cwd()


def load_tokens():
    path = find_project_root() / "style" / "hud_style_tokens.py"
    if not path.exists():
        return None

    spec = importlib.util.spec_from_file_location("hud_style_tokens", path)
    tokens = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(tokens)
    print(f"Loaded style tokens from: {path}")
    return tokens


def mpl_color(value):
    if isinstance(value, tuple) and len(value) == 3:
        return tuple(v / 255 for v in value)
    return value


tokens = load_tokens()

if THEME == "dark":
    BG = mpl_color(getattr(tokens, "SA_COLORS_BG", (2, 7, 13))) if tokens else "#02070d"
    TEXT = mpl_color(getattr(tokens, "SA_COLORS_TEXT_MAIN", (216, 251, 255))) if tokens else "#d8fbff"
    TEXT_DIM = mpl_color(getattr(tokens, "SA_COLORS_TEXT_DIM", (159, 199, 212))) if tokens else "#9fc7d4"
    GRID = mpl_color(getattr(tokens, "SA_CHART_GRID_COLOR", (42, 215, 255))) if tokens else "#2ad7ff"
    LINE_MAIN = mpl_color(getattr(tokens, "SA_CHART_LINE_COLOR", (90, 240, 255))) if tokens else "#5af0ff"
else:
    BG = "#f7fbff"
    TEXT = "#071a24"
    TEXT_DIM = "#33515c"
    GRID = "#8aa9b3"
    LINE_MAIN = "#007a9a"

STAR_COLORS = [
    "#5af0ff",
    "#52ff9a",
    "#ffd85a",
    "#ff9c52",
    "#c78cff",
    "#ff5f7f",
    "#8ffcff",
]

plt.rcParams.update({
    "text.usetex": False,
    "font.family": "DejaVu Sans Mono",
    "mathtext.fontset": "dejavusans",
    "figure.facecolor": BG,
    "axes.facecolor": BG,
    "savefig.facecolor": BG,
    "text.color": TEXT,
    "axes.labelcolor": TEXT_DIM,
    "xtick.color": TEXT_DIM,
    "ytick.color": TEXT_DIM,
    "axes.edgecolor": GRID,
})

# =========================================================
# DATA DISCOVERY / LOADING
# =========================================================

def discover_fits_files(data_dir: Path) -> list[Path]:
    files = sorted(data_dir.glob("*.fits")) + sorted(data_dir.glob("*.fit")) + sorted(data_dir.glob("*.fts"))
    if not files:
        raise FileNotFoundError(f"No FITS files found in {data_dir.resolve()}")
    return files


def guess_star_name(path: Path) -> str:
    name = path.stem.strip("_")

    replacements = {
        "gamcas": "Gamma Cassiopeiae",
        "zettau": "Zeta Tauri",
        "hd10516": "Phi Persei",
        "28tauha": "Pleione",
        "hd10144": "Achernar",
    }

    lower = name.lower()
    for key, value in replacements.items():
        if key in lower:
            return value

    return name.replace("_", " ")


def load_fits_spectrum(path: Path):
    with fits.open(path) as hdul:
        header = hdul[0].header
        data = hdul[0].data

        # Вариант 1: обычный 1D spectrum в primary HDU
        if data is not None:
            arr = np.asarray(data, dtype=float).squeeze()

            if arr.ndim == 1:
                flux = arr
                lambda_0 = header.get("CRVAL1", LAMBDA_0)
                delta = header.get("CDELT1", header.get("CD1_1", DELTA_LAMBDA))
                wavelength = lambda_0 + np.arange(len(flux)) * delta

            elif arr.ndim == 2:
                # Частый случай: [wavelength, flux] или [flux] x orders
                if arr.shape[0] == 2:
                    wavelength = arr[0]
                    flux = arr[1]
                elif arr.shape[1] == 2:
                    wavelength = arr[:, 0]
                    flux = arr[:, 1]
                else:
                    # Берем центральный order / row как fallback
                    row = arr.shape[0] // 2
                    flux = arr[row]
                    lambda_0 = header.get("CRVAL1", LAMBDA_0)
                    delta = header.get("CDELT1", header.get("CD1_1", DELTA_LAMBDA))
                    wavelength = lambda_0 + np.arange(len(flux)) * delta

            else:
                raise ValueError(f"Unsupported FITS data shape {arr.shape}: {path}")

        # Вариант 2: spectrum лежит в binary table extension
        elif len(hdul) > 1 and hdul[1].data is not None:

            table = hdul[1].data
            names = [n.lower() for n in table.names]

            wave_col = None
            flux_col = None

            # -----------------------------------------
            # wavelength column
            # -----------------------------------------

            for candidate in ["wavelength", "wave", "lambda", "lam"]:
                if candidate in names:
                    wave_col = table.names[names.index(candidate)]
                    break

            # -----------------------------------------
            # flux column
            # -----------------------------------------

            for candidate in ["flux", "spec", "spectrum", "intensity"]:
                if candidate in names:
                    flux_col = table.names[names.index(candidate)]
                    break

            if flux_col is None:
                raise ValueError(
                    f"Could not detect flux column in {path}. "
                    f"Columns: {table.names}"
                )

            # -----------------------------------------
            # flux
            # -----------------------------------------

            flux = np.asarray(
                table[flux_col],
                dtype=float
            ).squeeze()

            # -----------------------------------------
            # wavelength
            # -----------------------------------------

            if wave_col is not None:

                wavelength = np.asarray(
                    table[wave_col],
                    dtype=float
                ).squeeze()

            else:

                h0 = hdul[0].header
                h1 = hdul[1].header

                lambda_0 = (
                    h1.get("CRVAL1")
                    or h0.get("CRVAL1")
                    or LAMBDA_0
                )

                delta = (
                    h1.get("CDELT1")
                    or h1.get("CD1_1")
                    or h0.get("CDELT1")
                    or h0.get("CD1_1")
                    or DELTA_LAMBDA
                )

                wavelength = (
                    lambda_0
                    + np.arange(len(flux)) * delta
                )
                

        else:
            raise ValueError(f"No usable data found in FITS: {path}")

    wavelength = np.asarray(wavelength, dtype=float).squeeze()
    flux = np.asarray(flux, dtype=float).squeeze()

    if wavelength.ndim != 1 or flux.ndim != 1:
        raise ValueError(
            f"Could not reduce FITS to 1D wavelength/flux arrays: "
            f"{path}, wavelength={wavelength.shape}, flux={flux.shape}"
        )

    n = min(len(wavelength), len(flux))
    wavelength = wavelength[:n]
    flux = flux[:n]

    mask = (
        (wavelength >= WAVELENGTH_RANGE[0]) &
        (wavelength <= WAVELENGTH_RANGE[1]) &
        np.isfinite(wavelength) &
        np.isfinite(flux)
    )

    return wavelength[mask], flux[mask]


def normalize_flux(flux: np.ndarray) -> np.ndarray:
    med = np.nanmedian(flux)
    if not np.isfinite(med) or med == 0:
        return flux
    return flux / med


fits_files = discover_fits_files(DATA_DIR)

spectra = []
for idx, path in enumerate(fits_files):
    wavelength, flux = load_fits_spectrum(path)
    flux = normalize_flux(flux)

    if len(wavelength) < 5:
        continue

    spectra.append({
        "name": guess_star_name(path),
        "file": path,
        "wavelength": wavelength,
        "flux": flux,
        "color": STAR_COLORS[idx % len(STAR_COLORS)],
    })

if not spectra:
    raise RuntimeError("No usable spectra found.")

print(f"Loaded spectra: {len(spectra)}")
for s in spectra:
    print(f"- {s['name']}: {s['file']}")

# =========================================================
# COMMON RENDERING
# =========================================================

def setup_axis(ax, title: str, spectra_subset: list[dict]):
    all_w = np.concatenate([s["wavelength"] for s in spectra_subset])
    all_f = np.concatenate([s["flux"] for s in spectra_subset])

    ax.set_facecolor(BG)
    ax.set_xlim(np.nanmin(all_w), np.nanmax(all_w))

    ymin = np.nanmin(all_f)
    ymax = np.nanmax(all_f)
    margin = (ymax - ymin) * 0.12 if ymax > ymin else 0.1
    ax.set_ylim(ymin - margin, ymax + margin)

    ax.set_title(title, color=TEXT, pad=16, fontsize=15)
    ax.set_xlabel("Wavelength (Å)", color=TEXT_DIM)
    ax.set_ylabel("Normalized flux", color=TEXT_DIM)

    ax.grid(True, color=GRID, alpha=0.13, linewidth=0.7)

    for spine in ax.spines.values():
        spine.set_color(GRID)
        spine.set_alpha(0.45)

    ax.tick_params(colors=TEXT_DIM)


def render_spectrum_animation(spectra_subset: list[dict], out_file: Path, title: str):
    fig, ax = plt.subplots(figsize=(14, 7))
    setup_axis(ax, title, spectra_subset)

    lines = []
    for s in spectra_subset:
        line, = ax.plot([], [], lw=1.8, color=s["color"], label=s["name"])
        lines.append(line)

    legend = ax.legend(
        loc="upper right",
        frameon=False,
        fontsize=9,
        labelcolor=TEXT_DIM,
    )

    for text in legend.get_texts():
        text.set_color(TEXT_DIM)

    min_len = min(len(s["wavelength"]) for s in spectra_subset)

    def init():
        for line in lines:
            line.set_data([], [])
        return lines

    def update(frame):
        progress = frame / max(1, FRAMES - 1)

        for line, s in zip(lines, spectra_subset):
            w = s["wavelength"]
            f = s["flux"]

            if ANIMATION_STYLE == "left_to_right":
                n = max(2, int(progress * len(w)))
                line.set_data(w[:n], f[:n])

            elif ANIMATION_STYLE == "bottom_to_top":
                baseline = ax.get_ylim()[0]
                eased = progress * progress * (3 - 2 * progress)
                y = baseline + (f - baseline) * eased
                line.set_data(w, y)

            elif ANIMATION_STYLE == "alive":
                phase = 2 * np.pi * progress

                rng = np.random.default_rng(frame)

                noise = rng.normal(
                    loc=0.0,
                    scale=0.010,
                    size=len(f)
                )

                slow_gain = 1.0 + 0.004 * np.sin(phase * 2.0)

                y = f * slow_gain + noise

                line.set_data(w, y)


            else:
                raise ValueError(
                    "ANIMATION_STYLE must be 'left_to_right', 'bottom_to_top', or 'alive'"
                )

        return lines

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=FRAMES,
        init_func=init,
        interval=1000 / FPS,
        blit=True,
    )

    out_file.parent.mkdir(parents=True, exist_ok=True)

    if OUTPUT_FORMAT == "gif":
        writer = PillowWriter(fps=FPS)
        anim.save(out_file, writer=writer, dpi=DPI)

    elif OUTPUT_FORMAT == "mp4":
        writer = FFMpegWriter(
            fps=FPS,
            metadata=dict(artist="Stellar Attractor"),
            bitrate=2600,
        )
        anim.save(out_file, writer=writer, dpi=DPI)

    else:
        raise ValueError("OUTPUT_FORMAT must be 'gif' or 'mp4'")

    plt.close(fig)
    print(f"Saved: {out_file}")

# =========================================================
# EXPORT
# =========================================================

suffix = OUTPUT_FORMAT.lower()

if RENDER_MODE == "comparison":
    out = OUT_ROOT / f"comparative_spectrum_{ANIMATION_STYLE}_{THEME}.{suffix}"

    render_spectrum_animation(
        spectra,
        out,
        title="Comparative Spectrum of Be Stars — Hα Region",
    )

elif RENDER_MODE == "single_files":

    total = len(spectra)
    print(f"Rendering {total} spectra...")

    for idx, s in enumerate(spectra, start=1):

        safe_name = (
            s["name"]
            .lower()
            .replace(" ", "_")
            .replace("/", "_")
            .replace("\\", "_")
            .replace("—", "-")
            .replace("α", "alpha")
        )

        out = OUT_ROOT / f"{safe_name}_{ANIMATION_STYLE}_{THEME}.{suffix}"

        print(f"[{idx}/{total}] {s['name']} -> {out.name}")

        render_spectrum_animation(
            [s],
            out,
            title=f"{s['name']} Spectrum — Hα Region",
        )

        
else:
    raise ValueError("RENDER_MODE must be 'comparison' or 'single_files'")

Loaded style tokens from: /Users/mloktionov/PycharmProjects/Stellar_Attractor/ANIM/style/hud_style_tokens.py
Loaded spectra: 7
- Achernar HARPS cyan sp 0: data/Achernar_HARPS_cyan_sp_0.fits
- Achernar PUCHEROS optical sp 297: data/Achernar_PUCHEROS_optical_sp_297.fits
- Pleione: data/_28tauha_20250225_172a.fits
- Gamma Cassiopeiae: data/_gamcas_20250225_781.fits
- Achernar: data/_hd10144_20240913_317_A_Maetz , Ch_Kreider.fits
- Phi Persei: data/_hd10516_20210424_810_ChK.fits
- Zeta Tauri: data/_zettau_20250222_858.fits
Rendering 7 spectra...
[1/7] Achernar HARPS cyan sp 0 -> achernar_harps_cyan_sp_0_bottom_to_top_dark.gif
Saved: animations/spectra/achernar_harps_cyan_sp_0_bottom_to_top_dark.gif
[2/7] Achernar PUCHEROS optical sp 297 -> achernar_pucheros_optical_sp_297_bottom_to_top_dark.gif
Saved: animations/spectra/achernar_pucheros_optical_sp_297_bottom_to_top_dark.gif
[3/7] Pleione -> pleione_bottom_to_top_dark.gif
Saved: animations/spectra/pleione_bottom_to_top_dark.gif
[4/7] Gamma